# 01 · Ficheros del reto

Descarga del bucket `prc-2026-datasets` (OpenSky, solo lectura) los doce `training_2025-*.parquet`,
`ranking.parquet` (enero y julio de 2026) y `submitting.parquet` (la plantilla de 344.841 salidas) en `Data/raw/`.
En modo final baja además `final_ranking.parquet` (enero, febrero, junio y julio de 2026) y `final_submitting.parquet`
(670.790 salidas) en `Data/raw_final/`.

Necesita `credentials.json` en la raíz. Si los ficheros ya están, no descarga nada salvo que se fuerce.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "cuaderno.py").exists())
sys.path[:0] = [str(ROOT / "scripts"), str(ROOT)]
from cuaderno import elegir_periodo, existen, interruptor, run  # noqa: E402

PERIODO = elegir_periodo("ranking")   # abierto suelto: "ranking" (ene + jul 2026) o "final" (ene, feb, jun y jul 2026)
FINAL = PERIODO == "final"
import periodo as PER  # noqa: E402  rutas del periodo: PER.RANKING (fichero a predecir), PER.FEAT (variables)
from prc.base import config as K  # noqa: E402

DESCARGAR = interruptor("DESCARGAR_RETO", False)

In [ ]:
import duckdb

ficheros = sorted((ROOT / "Data" / "raw").glob("training_2025-*.parquet"))
completos = len(ficheros) == 12 and PER.RANKING.exists() and K.SUBMITTING_FILE.exists()
if DESCARGAR or not completos:
    run("scripts/descarga_reto.py")               # training_2025-*, ranking y submitting
    if FINAL:
        run("scripts/descarga_reto.py", "final")  # final_ranking y final_submitting
else:
    print("Ficheros del reto ya descargados: no se descarga nada")

In [ ]:
raw = (ROOT / "Data" / "raw").as_posix()
resumen = duckdb.sql(f"""
    select 'training 2025' fichero, count(*) filas, count(*) filter (where PHASE_mvt = 'DEP') salidas
    from '{raw}/training_*.parquet'
    union all select '{PER.RANKING.name}', count(*), count(*) filter (where PHASE_mvt = 'DEP') from '{PER.RANKING.as_posix()}'
    union all select '{K.SUBMITTING_FILE.name}', count(*), count(*) from '{K.SUBMITTING_FILE.as_posix()}'""").df()
assert resumen.set_index("fichero").loc[K.SUBMITTING_FILE.name, "filas"] == K.N_FILAS_ENVIO
resumen